# Advanced Retrieval with LangChain

In the following notebook, we'll explore various methods of advanced retrieval using LangChain!

We'll touch on:

- Naive Retrieval
- Best-Matching 25 (BM25)
- Multi-Query Retrieval
- Parent-Document Retrieval
- Contextual Compression (a.k.a. Rerank)
- Ensemble Retrieval
- Semantic chunking

We'll also discuss how these methods impact performance on our set of documents with a simple RAG chain.

There will be two breakout rooms:

- 🤝 Breakout Room Part #1
  - Task 1: Getting Dependencies!
  - Task 2: Data Collection and Preparation
  - Task 3: Setting Up QDrant!
  - Task 4-10: Retrieval Strategies
- 🤝 Breakout Room Part #2
  - Activity: Evaluate with Ragas

# 🤝 Breakout Room Part #1

## Task 1: Getting Dependencies!

We're going to need a few specific LangChain community packages, like OpenAI (for our [LLM](https://platform.openai.com/docs/models) and [Embedding Model](https://platform.openai.com/docs/guides/embeddings)) and Cohere (for our [Reranker](https://cohere.com/rerank)).

We'll also provide our OpenAI key, as well as our Cohere API key.

In [1]:
import os
import getpass

os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API Key:")

In [2]:
os.environ["COHERE_API_KEY"] = getpass.getpass("Cohere API Key:")

## Task 2: Data Collection and Preparation

We'll be using our Use Case Data once again - this time the strutured data available through the CSV!

### Data Preparation

We want to make sure all our documents have the relevant metadata for the various retrieval strategies we're going to be applying today.

In [3]:
#from langchain_community.document_loaders.csv_loader import CSVLoader
from datetime import datetime, timedelta

# loader = CSVLoader(
#     file_path=f"./data/Projects_with_Domains.csv",
#     metadata_columns=[
#       "Project Title",
#       "Project Domain",
#       "Secondary Domain",
#       "Description",
#       "Judge Comments",
#       "Score",
#       "Project Name",
#       "Judge Score"
#     ]
# )

#synthetic_usecase_data = loader.load()

#for doc in synthetic_usecase_data:
    #doc.page_content = doc.metadata["Description"]

Let's look at an example document to see if everything worked as expected!

In [5]:
#synthetic_usecase_data[0]

Start with one PDF to verify parsing, then expand to the full folder later.

In [6]:
# --- Single-file test run ---
#TODO get multiple files 
#pdf_paths = glob.glob("./kids_science_pdfs/*.pdf")
pdf_path = "data/grade3/Grade 3 Science Booklets (Ontario Curriculum Aligned).pdf"  # change later
loader = PyPDFLoader(pdf_path)
pdf_docs = loader.load()

filename = os.path.basename(pdf_path)
parts = filename.replace(".pdf", "").split("_")

#TODO Make it dynamic for all the files
#doc_type = "Fact" if parts[0] == "F" else "Concept"
doc_type = "Concept"
#grade = parts[1] if len(parts) > 1 else "Unknown"
grade = "3"

#topic = " ".join(parts[2:]) if len(parts) > 2 else "General"
topic = "Simple Machines"

docs = []
for d in pdf_docs:
    d.metadata.update({
        "grade": grade,
        "type": doc_type,
        "topic": topic,
        "source": filename,
        "strand": "TBD"  # later tag: Life / Energy / Structures / Earth
    })
    docs.append(d)

print(f"Loaded {len(docs)} chunks from {filename}")
print("Sample metadata:", docs[0].metadata)
print("Sample content:", docs[0].page_content[:250])

NameError: name 'PyPDFLoader' is not defined

Block 2 — Re-chunk the text (give RAGAS enough context)

In [91]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=2000, chunk_overlap=200)
docs = splitter.split_documents(docs)

print(f"✅ After re-split: {len(docs)} chunks")
print("Preview:", docs[0].page_content[:200])

✅ After re-split: 49 chunks
Preview: Grade 3 Science Booklets (Ontario Curriculum
Aligned)
Below are outlines for five Grade 3 science booklets, each aligned with the Ontario Science & Technology
curriculum  (TVO  Learn).  Four  booklets


Block 3 — Build & inspect the Knowledge Graph

In [96]:
from ragas.testset.graph import KnowledgeGraph, Node, NodeType

# Create empty KG
kg = KnowledgeGraph()

# Add one node per LangChain Document
for doc in docs:
    kg.nodes.append(
        Node(
            type=NodeType.DOCUMENT,
            properties={
                "page_content": doc.page_content,
                "metadata": doc.metadata
            }
        )
    )

print(f"📘 KG manually built with {len(kg.nodes)} nodes and {len(kg.relationships)} relationships")

📘 KG manually built with 49 nodes and 0 relationships


Block 3.5 — Add Summaries to Knowledge Graph (RAGAS Persona Requirement)

In [102]:
# --- Block 3.5: Add placeholder summaries ---
for node in kg.nodes:
    text = (node.properties.get("page_content") or "")[:600]
    node.properties["summary"] = text if text else "Summary unavailable."
print("✅ Added placeholder summaries to", len(kg.nodes), "nodes.")

# --- Patch: verify all summaries are non-empty strings (RAGAS filter fix) ---
for node in kg.nodes:
    s = node.properties.get("summary")
    if not isinstance(s, str) or len(s.strip()) == 0:
        node.properties["summary"] = "This is a summary of a Grade 3 science text."
print("✅ Verified summaries: all nodes now have valid strings.")

# --- Continue to Block 4: Generate Golden Dataset ---
from ragas.testset import TestsetGenerator

generator = TestsetGenerator(
    llm=chat_model,
    embedding_model=embeddings,
    knowledge_graph=kg
)

dataset = generator.generate(testset_size=10)
df = dataset.to_pandas()
df.head()

dataset.save("golden_dataset_science.json")
print("✅ Golden dataset created and saved.")

✅ Added placeholder summaries to 49 nodes.
✅ Verified summaries: all nodes now have valid strings.


ValueError: No nodes that satisfied the given filer. Try changing the filter.

Block 4 — Generate golden dataset (synthetic Q&A)

In [100]:
from ragas.testset import TestsetGenerator

# Attach the KG directly at init
generator = TestsetGenerator(
    llm=chat_model,
    embedding_model=embeddings,
    knowledge_graph=kg
)

# Generate dataset (no transforms arg here)
dataset = generator.generate(testset_size=10)

df = dataset.to_pandas()
df.head()

dataset.save("golden_dataset_science.json")
print("✅ Golden dataset created and saved.")

ValueError: No nodes that satisfied the given filer. Try changing the filter.

## Task 3: Setting up QDrant!

Now that we have our documents, let's create a QDrant VectorStore with the collection name "Synthetic_Usecases".

We'll leverage OpenAI's [`text-embedding-3-small`](https://openai.com/blog/new-embedding-models-and-api-updates) because it's a very powerful (and low-cost) embedding model.

> NOTE: We'll be creating additional vectorstores where necessary, but this pattern is still extremely useful.

In [79]:
from langchain_community.vectorstores import Qdrant
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

vectorstore = Qdrant.from_documents(
    synthetic_usecase_data,
    embeddings,
    location=":memory:",
    collection_name="Synthetic_Usecases"
)

## Task 4: Naive RAG Chain

Since we're focusing on the "R" in RAG today - we'll create our Retriever first.

### R - Retrieval

This naive retriever will simply look at each review as a document, and use cosine-similarity to fetch the 10 most relevant documents.

> NOTE: We're choosing `10` as our `k` here to provide enough documents for our reranking process later

In [80]:
naive_retriever = vectorstore.as_retriever(search_kwargs={"k" : 10})

### A - Augmented

We're going to go with a standard prompt for our simple RAG chain today! Nothing fancy here, we want this to mostly be about the Retrieval process.

In [81]:
from langchain_core.prompts import ChatPromptTemplate

RAG_TEMPLATE = """\
You are a helpful and kind assistant. Use the context provided below to answer the question.

If you do not know the answer, or are unsure, say you don't know.

Query:
{question}

Context:
{context}
"""

rag_prompt = ChatPromptTemplate.from_template(RAG_TEMPLATE)

### G - Generation

We're going to leverage `gpt-4.1-nano` as our LLM today, as - again - we want this to largely be about the Retrieval process.

In [82]:
from langchain_openai import ChatOpenAI

chat_model = ChatOpenAI(model="gpt-4.1-nano")

In [87]:
len(docs)
docs[0].metadata

{'producer': 'WeasyPrint 65.1',
 'creator': 'ChatGPT',
 'creationdate': '',
 'title': 'Grade 3 Science Booklets (Ontario Curriculum Aligned)',
 'author': 'ChatGPT Deep Research',
 'source': 'Grade 3 Science Booklets (Ontario Curriculum Aligned).pdf',
 'total_pages': 25,
 'page': 0,
 'page_label': '1',
 'grade': '3',
 'type': 'Concept',
 'topic': 'Simple Machines',
 'strand': 'TBD'}

In [88]:
# --- Notebook Checkpoint ---
def verify_environment():
    checks = {}

    # 1️⃣ Verify documents
    try:
        checks['docs_count'] = len(docs)
        checks['sample_metadata'] = docs[0].metadata
        assert checks['docs_count'] > 0, "No docs loaded — rerun your PDF loader cell."
    except Exception as e:
        checks['docs_error'] = str(e)

    # 2️⃣ Verify LLM
    try:
        _ = chat_model.model  # Access model name
        checks['chat_model'] = str(chat_model)
    except Exception as e:
        checks['chat_model_error'] = str(e)

    # 3️⃣ Verify embeddings
    try:
        _ = embeddings.model
        checks['embeddings_model'] = str(embeddings)
    except Exception as e:
        checks['embeddings_error'] = str(e)

    print("✅ Environment Check Results:")
    for k, v in checks.items():
        print(f"{k}: {v}")

verify_environment()

✅ Environment Check Results:
docs_count: 25
sample_metadata: {'producer': 'WeasyPrint 65.1', 'creator': 'ChatGPT', 'creationdate': '', 'title': 'Grade 3 Science Booklets (Ontario Curriculum Aligned)', 'author': 'ChatGPT Deep Research', 'source': 'Grade 3 Science Booklets (Ontario Curriculum Aligned).pdf', 'total_pages': 25, 'page': 0, 'page_label': '1', 'grade': '3', 'type': 'Concept', 'topic': 'Simple Machines', 'strand': 'TBD'}
chat_model_error: 'ChatOpenAI' object has no attribute 'model'
embeddings_model: client=<openai.resources.embeddings.Embeddings object at 0x7b557ca1b490> async_client=<openai.resources.embeddings.AsyncEmbeddings object at 0x7b557ca1bbb0> model='text-embedding-3-small' dimensions=None deployment='text-embedding-ada-002' openai_api_version=None openai_api_base=None openai_api_type=None openai_proxy=None embedding_ctx_length=8191 openai_api_key=SecretStr('**********') openai_organization=None allowed_special=None disallowed_special=None chunk_size=1000 max_retrie

Reuse  Session 7 Synthetic Data Generator to make ~30 Q-A pairs.

In [90]:
from ragas.testset import TestsetGenerator

# Instantiate generator using same structure as Session 7
generator = TestsetGenerator(llm=chat_model, embedding_model=embeddings)

# Generate golden dataset from Kids Science PDFs
dataset = generator.generate_with_langchain_docs(
    docs,
    testset_size=10    # start small; scale later
)

# View sample
df = dataset.to_pandas()
df.head()

Applying HeadlinesExtractor:   0%|          | 0/24 [00:00<?, ?it/s]

unable to apply transformation: 'str' object has no attribute 'content'
unable to apply transformation: 'str' object has no attribute 'content'
unable to apply transformation: 'str' object has no attribute 'content'
unable to apply transformation: 'str' object has no attribute 'content'
unable to apply transformation: 'str' object has no attribute 'content'
unable to apply transformation: 'str' object has no attribute 'content'
unable to apply transformation: 'str' object has no attribute 'content'
unable to apply transformation: 'str' object has no attribute 'content'
unable to apply transformation: 'str' object has no attribute 'content'
unable to apply transformation: 'str' object has no attribute 'content'
unable to apply transformation: 'str' object has no attribute 'content'
unable to apply transformation: 'str' object has no attribute 'content'
unable to apply transformation: 'str' object has no attribute 'content'
unable to apply transformation: 'str' object has no attribute 'c

Applying HeadlineSplitter:   0%|          | 0/25 [00:00<?, ?it/s]

unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to ap

Applying SummaryExtractor:   0%|          | 0/24 [00:00<?, ?it/s]

...

Applying [EmbeddingExtractor, ThemesExtractor, NERExtractor]:   0%|          | 0/24 [00:00<?, ?it/s]

unable to apply transformation: node.property('summary') must be a string, found '<class 'NoneType'>'
unable to apply transformation: node.property('summary') must be a string, found '<class 'NoneType'>'
unable to apply transformation: node.property('summary') must be a string, found '<class 'NoneType'>'
unable to apply transformation: node.property('summary') must be a string, found '<class 'NoneType'>'
unable to apply transformation: node.property('summary') must be a string, found '<class 'NoneType'>'
unable to apply transformation: node.property('summary') must be a string, found '<class 'NoneType'>'
unable to apply transformation: node.property('summary') must be a string, found '<class 'NoneType'>'
unable to apply transformation: node.property('summary') must be a string, found '<class 'NoneType'>'
unable to apply transformation: node.property('summary') must be a string, found '<class 'NoneType'>'
unable to apply transformation: node.property('summary') must be a string, found '

Applying [CosineSimilarityBuilder, OverlapScoreBuilder]:   0%|          | 0/2 [00:00<?, ?it/s]

unable to apply transformation: Node 6c671a63-9877-462e-957d-b3fcbfc7dee0 has no summary_embedding


ValueError: No nodes that satisfied the given filer. Try changing the filter.

### LCEL RAG Chain

We're going to use LCEL to construct our chain.

> NOTE: This chain will be exactly the same across the various examples with the exception of our Retriever!

In [10]:
from langchain_core.runnables import RunnablePassthrough
from operator import itemgetter
from langchain_core.output_parsers import StrOutputParser

naive_retrieval_chain = (
    # INVOKE CHAIN WITH: {"question" : "<<SOME USER QUESTION>>"}
    # "question" : populated by getting the value of the "question" key
    # "context"  : populated by getting the value of the "question" key and chaining it into the base_retriever
    {"context": itemgetter("question") | naive_retriever, "question": itemgetter("question")}
    # "context"  : is assigned to a RunnablePassthrough object (will not be called or considered in the next step)
    #              by getting the value of the "context" key from the previous step
    | RunnablePassthrough.assign(context=itemgetter("context"))
    # "response" : the "context" and "question" values are used to format our prompt object and then piped
    #              into the LLM and stored in a key called "response"
    # "context"  : populated by getting the value of the "context" key from the previous step
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's see how this simple chain does on a few different prompts.

> NOTE: You might think that we've cherry picked prompts that showcase the individual skill of each of the retrieval strategies - you'd be correct!

In [11]:
naive_retrieval_chain.invoke({"question" : "What is the most common project domain?"})["response"].content

'The most common project domain in the provided data is "Healthcare / MedTech," which appears three times among the listed projects.'

In [12]:
naive_retrieval_chain.invoke({"question" : "Were there any usecases about security?"})["response"].content

'Yes, there are usecases related to security. Specifically, one project titled "Pathfinder 24" in the Healthcare / MedTech domain has a secondary focus on "Security."'

In [13]:
naive_retrieval_chain.invoke({"question" : "What did judges have to say about the fintech projects?"})["response"].content

'Judges generally had positive comments about the fintech projects. For example, one project was described as having a "measurable environmental benefit," another as "impressive real-world impact," and a different one as "robust experimental validation." Additionally, some projects were noted for their "strong quantitative results" and "excellent code quality." Overall, the judges recognized the projects as innovative, impactful, and well-executed, though some noted minor issues with integration or the need for additional benchmarking.'

Overall, this is not bad! Let's see if we can make it better!

## Task 5: Best-Matching 25 (BM25) Retriever

Taking a step back in time - [BM25](https://www.nowpublishers.com/article/Details/INR-019) is based on [Bag-Of-Words](https://en.wikipedia.org/wiki/Bag-of-words_model) which is a sparse representation of text.

In essence, it's a way to compare how similar two pieces of text are based on the words they both contain.

This retriever is very straightforward to set-up! Let's see it happen down below!


In [14]:
from langchain_community.retrievers import BM25Retriever

bm25_retriever = BM25Retriever.from_documents(synthetic_usecase_data)

We'll construct the same chain - only changing the retriever.

In [15]:
bm25_retrieval_chain = (
    {"context": itemgetter("question") | bm25_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's look at the responses!

In [16]:
bm25_retrieval_chain.invoke({"question" : "What is the most common project domain?"})["response"].content

'The most common project domain is not explicitly identified in the provided data. Based on the sample entries, the projects belong to various domains such as Productivity Assistants, E‑commerce / Marketplaces, Healthcare / MedTech, and Finance / FinTech. Without additional data on the total number of projects in each domain, I cannot determine which is the most common.'

In [17]:
bm25_retrieval_chain.invoke({"question" : "Were there any usecases about security?"})["response"].content

'Yes, there is a use case related to security. The project titled "SecureNest 49" focuses on a document summarization and retrieval system for enterprise knowledge bases, which is in the domain of E‑commerce / Marketplaces and Legal / Compliance.'

In [18]:
bm25_retrieval_chain.invoke({"question" : "What did judges have to say about the fintech projects?"})["response"].content

'The judges described the fintech project, "SynthMind," as having a conceptually strong approach but noted that its results need more benchmarking.'

It's not clear that this is better or worse, if only we had a way to test this (SPOILERS: We do, the second half of the notebook will cover this)

#### ❓ Question #1:

Give an example query where BM25 is better than embeddings and justify your answer.

##### ✅ Answer


## Task 6: Contextual Compression (Using Reranking)

Contextual Compression is a fairly straightforward idea: We want to "compress" our retrieved context into just the most useful bits.

There are a few ways we can achieve this - but we're going to look at a specific example called reranking.

The basic idea here is this:

- We retrieve lots of documents that are very likely related to our query vector
- We "compress" those documents into a smaller set of *more* related documents using a reranking algorithm.

We'll be leveraging Cohere's Rerank model for our reranker today!

All we need to do is the following:

- Create a basic retriever
- Create a compressor (reranker, in this case)

That's it!

Let's see it in the code below!

In [19]:
from langchain.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain_cohere import CohereRerank

compressor = CohereRerank(model="rerank-v3.5")
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor, base_retriever=naive_retriever
)

Let's create our chain again, and see how this does!

In [20]:
contextual_compression_retrieval_chain = (
    {"context": itemgetter("question") | compression_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

In [21]:
contextual_compression_retrieval_chain.invoke({"question" : "What is the most common project domain?"})["response"].content

'The most common project domain, based on the provided data, appears to be "Security," as it is listed in at least one of the records. However, given that only a few data points are provided here, I cannot conclusively determine the most common domain overall. If the dataset contains more entries, the most common domain could be different.'

In [22]:
contextual_compression_retrieval_chain.invoke({"question" : "Were there any usecases about security?"})["response"].content

'Based on the provided information, there are no explicit use cases related to security. The projects mentioned focus on privacy improvements in healthcare applications using federated learning, but there is no specific mention of security-related use cases.'

In [23]:
contextual_compression_retrieval_chain.invoke({"question" : "What did judges have to say about the fintech projects?"})["response"].content

'The judges had varying comments about the fintech projects. Specifically, for the project "Pathfinder 27" in the Finance/FinTech domain, the judge praised its "Excellent code quality and use of open-source libraries," and awarded a high score of 9.8. However, the provided context does not include specific judge comments about other fintech projects.'

We'll need to rely on something like Ragas to help us get a better sense of how this is performing overall - but it "feels" better!

## Task 7: Multi-Query Retriever

Typically in RAG we have a single query - the one provided by the user.

What if we had....more than one query!

In essence, a Multi-Query Retriever works by:

1. Taking the original user query and creating `n` number of new user queries using an LLM.
2. Retrieving documents for each query.
3. Using all unique retrieved documents as context

So, how is it to set-up? Not bad! Let's see it down below!



In [24]:
from langchain.retrievers.multi_query import MultiQueryRetriever

multi_query_retriever = MultiQueryRetriever.from_llm(
    retriever=naive_retriever, llm=chat_model
) 

In [25]:
multi_query_retrieval_chain = (
    {"context": itemgetter("question") | multi_query_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

In [26]:
multi_query_retrieval_chain.invoke({"question" : "What is the most common project domain?"})["response"].content

'Based on the provided data, the most common project domain appears to be "Healthcare / MedTech," as it is mentioned multiple times among the projects listed.'

In [27]:
multi_query_retrieval_chain.invoke({"question" : "Were there any usecases about security?"})["response"].content

'Yes, there are use cases related to security. Specifically, one of the projects is titled "MediMind," which falls under the Security domain. The description indicates it is a medical imaging solution aimed at improving early diagnosis through vision transformers, and it was noted for exceeding expectations in creativity and usability.'

In [28]:
multi_query_retrieval_chain.invoke({"question" : "What did judges have to say about the fintech projects?"})["response"].content

'Judges had a generally positive outlook on the fintech projects. For example, the project named "Pathfinder 27" received high praise for its excellent code quality and effective use of open-source libraries, with a judge score of 9.8. "SecureNest 28" was considered conceptually strong, though it needed more benchmarking, and scored 9.0. Overall, the judges appreciated the technical quality, scalability, and real-world impact of these fintech projects, though some noted the need for further benchmarking and qualitative analysis.'

#### ❓ Question #2:

Explain how generating multiple reformulations of a user query can improve recall.

##### ✅ Answer


## Task 8: Parent Document Retriever

A "small-to-big" strategy - the Parent Document Retriever works based on a simple strategy:

1. Each un-split "document" will be designated as a "parent document" (You could use larger chunks of document as well, but our data format allows us to consider the overall document as the parent chunk)
2. Store those "parent documents" in a memory store (not a VectorStore)
3. We will chunk each of those documents into smaller documents, and associate them with their respective parents, and store those in a VectorStore. We'll call those "child chunks".
4. When we query our Retriever, we will do a similarity search comparing our query vector to the "child chunks".
5. Instead of returning the "child chunks", we'll return their associated "parent chunks".

Okay, maybe that was a few steps - but the basic idea is this:

- Search for small documents
- Return big documents

The intuition is that we're likely to find the most relevant information by limiting the amount of semantic information that is encoded in each embedding vector - but we're likely to miss relevant surrounding context if we only use that information.

Let's start by creating our "parent documents" and defining a `RecursiveCharacterTextSplitter`.

In [29]:
from langchain.retrievers import ParentDocumentRetriever
from langchain.storage import InMemoryStore
from langchain_text_splitters import RecursiveCharacterTextSplitter
from qdrant_client import QdrantClient, models

parent_docs = synthetic_usecase_data
child_splitter = RecursiveCharacterTextSplitter(chunk_size=750)

We'll need to set up a new QDrant vectorstore - and we'll use another useful pattern to do so!

> NOTE: We are manually defining our embedding dimension, you'll need to change this if you're using a different embedding model.

In [30]:
from langchain_qdrant import QdrantVectorStore

client = QdrantClient(location=":memory:")

client.create_collection(
    collection_name="full_documents",
    vectors_config=models.VectorParams(size=1536, distance=models.Distance.COSINE)
)

parent_document_vectorstore = QdrantVectorStore(
    collection_name="full_documents", embedding=OpenAIEmbeddings(model="text-embedding-3-small"), client=client
)

Now we can create our `InMemoryStore` that will hold our "parent documents" - and build our retriever!

In [31]:
store = InMemoryStore()

parent_document_retriever = ParentDocumentRetriever(
    vectorstore = parent_document_vectorstore,
    docstore=store,
    child_splitter=child_splitter,
)

By default, this is empty as we haven't added any documents - let's add some now!

In [32]:
parent_document_retriever.add_documents(parent_docs, ids=None)

We'll create the same chain we did before - but substitute our new `parent_document_retriever`.

In [33]:
parent_document_retrieval_chain = (
    {"context": itemgetter("question") | parent_document_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's give it a whirl!

In [34]:
parent_document_retrieval_chain.invoke({"question" : "What is the most common project domain?"})["response"].content

'Based on the provided data, the project domains mentioned are Security, Creative / Design / Media, Healthcare / MedTech, and Productivity Assistants. Since there are only a few examples, it is difficult to determine the most common domain definitively from this small sample. However, given the pattern, if you have access to the full dataset, the most common project domain would be whichever domain appears most frequently across all entries.\n\nFrom the small sample provided, no single domain clearly dominates. If you need a precise answer, please refer to the complete dataset.'

In [35]:
parent_document_retrieval_chain.invoke({"question" : "Were there any usecases about security?"})["response"].content

'Based on the provided context, there are no specific use cases related to security mentioned. The primary focus appears to be on federated learning toolkits aimed at improving privacy in healthcare applications.'

In [36]:
parent_document_retrieval_chain.invoke({"question" : "What did judges have to say about the fintech projects?"})["response"].content

'The judges had positive comments about the fintech projects, describing them as technically ambitious, well-executed, and offering clever solutions with measurable benefits. For example, one project was praised as a "comprehensive and technically mature approach," another as "technically ambitious and well-executed," and a third as a "clever solution with measurable environmental benefit."'

Overall, the performance *seems* largely the same. We can leverage a tool like [Ragas]() to more effectively answer the question about the performance.

## Task 9: Ensemble Retriever

In brief, an Ensemble Retriever simply takes 2, or more, retrievers and combines their retrieved documents based on a rank-fusion algorithm.

In this case - we're using the [Reciprocal Rank Fusion](https://plg.uwaterloo.ca/~gvcormac/cormacksigir09-rrf.pdf) algorithm.

Setting it up is as easy as providing a list of our desired retrievers - and the weights for each retriever.

In [37]:
from langchain.retrievers import EnsembleRetriever

retriever_list = [bm25_retriever, naive_retriever, parent_document_retriever, compression_retriever, multi_query_retriever]
equal_weighting = [1/len(retriever_list)] * len(retriever_list)

ensemble_retriever = EnsembleRetriever(
    retrievers=retriever_list, weights=equal_weighting
)

We'll pack *all* of these retrievers together in an ensemble.

In [38]:
ensemble_retrieval_chain = (
    {"context": itemgetter("question") | ensemble_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's look at our results!

In [39]:
ensemble_retrieval_chain.invoke({"question" : "What is the most common project domain?"})["response"].content

'Based on the provided data, the most common project domain appears to be "Finance / FinTech," which is mentioned twice among the projects. However, since the dataset sample is limited, I cannot confirm if it is the most frequent overall. \n\nFrom the sample, "Finance / FinTech" is notably recurring, but for a definitive answer, a complete count across the entire dataset would be necessary.'

In [40]:
ensemble_retrieval_chain.invoke({"question" : "Were there any usecases about security?"})["response"].content

'Yes, there is a use case related to security. Specifically, the project "SecureNest 49" is a document summarization and retrieval system for enterprise knowledge bases, which involves elements of security and compliance. Additionally, the project "PlanPilot 35" involves a federated learning toolkit aimed at improving privacy in healthcare applications, which also relates to security concerns around data privacy.'

In [41]:
ensemble_retrieval_chain.invoke({"question" : "What did judges have to say about the fintech projects?"})["response"].content

'The judges had varied comments about the fintech projects. For "PlanPilot 35," they described it as "a clever solution with measurable environmental benefit." For "SynthMind," they considered it a "conceptually strong" platform, though noting that "results need more benchmarking." Overall, the judges recognized the ideas as promising but emphasized the need for stronger evaluation metrics or benchmarking in some cases.'

## Task 10: Semantic Chunking

While this is not a retrieval method - it *is* an effective way of increasing retrieval performance on corpora that have clean semantic breaks in them.

Essentially, Semantic Chunking is implemented by:

1. Embedding all sentences in the corpus.
2. Combining or splitting sequences of sentences based on their semantic similarity based on a number of [possible thresholding methods](https://python.langchain.com/docs/how_to/semantic-chunker/):
  - `percentile`
  - `standard_deviation`
  - `interquartile`
  - `gradient`
3. Each sequence of related sentences is kept as a document!

Let's see how to implement this!

We'll use the `percentile` thresholding method for this example which will:

Calculate all distances between sentences, and then break apart sequences of setences that exceed a given percentile among all distances.

In [42]:
from langchain_experimental.text_splitter import SemanticChunker

semantic_chunker = SemanticChunker(
    embeddings,
    breakpoint_threshold_type="percentile"
)

Now we can split our documents.

In [43]:
semantic_documents = semantic_chunker.split_documents(synthetic_usecase_data[:20])

Let's create a new vector store.

In [44]:
semantic_vectorstore = Qdrant.from_documents(
    semantic_documents,
    embeddings,
    location=":memory:",
    collection_name="Synthetic_Usecase_Data_Semantic_Chunks"
)

We'll use naive retrieval for this example.

In [45]:
semantic_retriever = semantic_vectorstore.as_retriever(search_kwargs={"k" : 10})

Finally we can create our classic chain!

In [46]:
semantic_retrieval_chain = (
    {"context": itemgetter("question") | semantic_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

And view the results!

In [47]:
semantic_retrieval_chain.invoke({"question" : "What is the most common project domain?"})["response"].content

'Based on the provided data, the most common project domain appears to be "Legal / Compliance," which is mentioned twice. Other domains like "Developer Tools / DevEx" and "Customer Support / Helpdesk" are also mentioned multiple times, but "Legal / Compliance" has the highest frequency in this subset. However, since the dataset might be larger, I cannot definitively say it\'s the most common overall, but from this sample, "Legal / Compliance" seems to be the most common.\n'

In [48]:
semantic_retrieval_chain.invoke({"question" : "Were there any usecases about security?"})["response"].content

'Yes, there are use cases related to security. Specifically, one project titled "SynthMind" in the "Developer Tools / DevEx" domain focuses on a medical imaging solution improving early diagnosis through vision transformers, with a secondary domain of Security. Additionally, another project named "SecureNest" in the "Security" domain involves a low-latency inference system for multimodal agents in autonomous systems.'

In [49]:
semantic_retrieval_chain.invoke({"question" : "What did judges have to say about the fintech projects?"})["response"].content

'Judges had various opinions about the fintech projects. For example:\n\n- The project titled "TrendLens 19" was described as "Technically ambitious and well-executed."\n- "WealthifyAI 16" was also noted for being "Comprehensive and technically mature approach."\n- "AutoMate 5" received praise for being "A forward-looking idea with solid supporting data."\n- "InsightAI 1" was highlighted as "Technically ambitious and well-executed."\n\nOverall, the judges recognized the fintech projects as ambitious, well-executed, and possessing solid technical foundations.'

#### ❓ Question #3:

If sentences are short and highly repetitive (e.g., FAQs), how might semantic chunking behave, and how would you adjust the algorithm?

##### ✅ Answer


# 🤝 Breakout Room Part #2

#### 🏗️ Activity #1

Your task is to evaluate the various Retriever methods against eachother.

You are expected to:

1. Create a "golden dataset"
 - Use Synthetic Data Generation (powered by Ragas, or otherwise) to create this dataset
2. Evaluate each retriever with *retriever specific* Ragas metrics
 - Semantic Chunking is not considered a retriever method and will not be required for marks, but you may find it useful to do a "semantic chunking on" vs. "semantic chunking off" comparision between them
3. Compile these in a list and write a small paragraph about which is best for this particular data and why.

Your analysis should factor in:
  - Cost
  - Latency
  - Performance

> NOTE: This is **NOT** required to be completed in class. Please spend time in your breakout rooms creating a plan before moving on to writing code.

##### HINTS:

- LangSmith provides detailed information about latency and cost.

Heres notes: 

1.) Step 1: Creating the golden dataset
    
using ragas.testset --> SDG--> import ragas, create knowledge graph, set up embedding knowledge (relationships)--> cohere reranker --> chunking method--> the data itself
what kind of questions do we want to use (what is the distibution for test questions) and how many questions


2.) Step 2: evaluate each retriever 
    
test each retriever method on the golden data set (evaluate it)--> Context Precision
          Context Recall
          Context Entities Recall
          Noise Sensitivity
          Response Relevancy
          Faithfulness
          Multimodal Faithfulness
          Multimodal Relevance
  ----> determine which one is relevant to which 
use metrics from LangSmith - latency, correctnes? token? use? other custom evaluators like "dopeness"?

    
need to build a rag pipeline --> evaluate it with ragas, LangSmith 🙂
